*Companion notebook for* **Social and Media Platform APIs**, *from* [Web Data Science](https://cuinfoscience.github.io/Web-Data-Science-Book/) *by Brian C. Keegan (INFO 4617/5617, University of Colorado Boulder).*

*Generated from `ch-12-social.qmd` — the book chapter is the authoritative version. Code cells are provided unexecuted: run them yourself, and expect to install the chapter's libraries and supply your own API keys where noted. Licensed CC BY-NC-SA 4.0.*

# Social and Media Platform APIs

## Learning Objectives
- Explain why a platform's API, unlike a government agency's, exists at the platform's discretion, and date what you collect accordingly
- Check who an account belongs to before you collect its posts: a domain handle, a verification, or neither
- Read profiles and posts from Bluesky's public API with no account or key, and page through more posts than one request returns with a `cursor`
- Read a channel's videos and their statistics from the YouTube Data API with a key, and stay inside its daily quota
- Evaluate the archives that replaced Pushshift for Reddit data, and what each one keeps that its users deleted
- Keep private people out of what you collect, save, and show

## Platforms, Data, and Power

The government APIs in @sec-government exist because a law says the data is public. A social platform's API exists because the company that runs the platform chooses to offer it, and the company can choose again. Since 2023, most of them have:

- **Reddit** priced heavy use of its API in 2023 and closed self-service access on November 11, 2025: a new app now needs Reddit's approval ([Reddit's announcement](https://www.reddit.com/r/redditdev/comments/1oug31u/introducing_the_responsible_builder_policy_new/)).
- **Spotify** removed its audio features from new apps in November 2024, and since February 11, 2026, every new developer app's owner needs a Premium subscription, with top tracks, popularity, and follower counts removed ([Spotify's migration guide](https://developer.spotify.com/documentation/web-api/tutorials/february-2026-migration-guide)).
- **X**, formerly Twitter, ended free research access in 2023 [@davidson2023platform].

Two platforms in this chapter answer differently. **Bluesky** is built on an open protocol, the AT Protocol, and its public API answers anyone, with no account and no key. **YouTube**'s Data API asks for a key, which anyone with a Google account can make in a few minutes, and counts what each key uses against a daily quota. Between them, they show two ways an API can be open: to anyone, or to anyone who identifies themselves.

| Platform | Who can read, as of November 2026 | What it asks of you | This chapter |
|---|---|---|---|
| Bluesky | Anyone | Nothing; a User-Agent, as always | Code |
| YouTube | Anyone with a Google account | A key, and a daily quota | Code |
| Reddit | Approved apps; archives of past posts | Reddit's approval, or an archive's terms | An archive's code, and history |
| Spotify | App owners with Premium | A subscription, and fewer endpoints | History |

Every claim in this chapter about a platform's rules is dated, because they change. When you collect from a platform, record the date, the endpoint, and what access you had, so a reader can tell what you could and couldn't see.

### Social Data Is People Data

Almost everything a platform's API returns was made by a person, most of whom never imagined it in a research dataset. Before collecting, revisit @sec-ethics: read the platform's terms of service, consider whether your project needs a review by your institution's IRB, collect the minimum you need, and never collect private messages. When you report what you found, don't quote a post if a search engine could lead a reader to its author [@ayers2018dont].

This chapter's running case avoids most of that: it reads **federal agencies**, public institutions that post in order to be read. Even so, an agency's posts can carry other people's: a repost shares someone else's post, and replies come from anyone. The code below sets those aside, and no private person's handle or post appears in any output.

## Missing Manual Reference
For storing and loading API keys safely, with environment variables, `.env` files, and what belongs in `.gitignore`, see the *Missing Manual*'s [Chapter 34: Environment Variables and Secrets](https://cuinfoscience.github.io/INFO-Missing-Manual/chapters/secrets.html).

## A Tour: From a Platform's Website to Its API

As in @sec-government, start with the page a person reads. Open the National Weather Service's profile on Bluesky's website, [bsky.app/profile/nws.noaa.gov](https://bsky.app/profile/nws.noaa.gov), and NOAA's channel on YouTube, [youtube.com/@NOAA](https://www.youtube.com/@NOAA). Every number on those pages comes from the platform's API, and each feature of the page has a request behind it:

| On the page | Bluesky | YouTube |
|---|---|---|
| The account's name and handle | `app.bsky.actor.getProfile`: `displayName`, `handle` | `channels.list`: `snippet.title`, `snippet.customUrl` |
| Proof the account is the agency's | The `.gov` handle; `verification` | The channel linked from the agency's own website |
| Followers or subscribers | `followersCount` | `statistics.subscriberCount` |
| The list of posts or videos | `app.bsky.feed.getAuthorFeed`, newest first | `playlistItems.list` on the channel's uploads playlist |
| Likes, reposts, views on each item | `likeCount`, `repostCount`, `replyCount` | `videos.list`: `statistics.viewCount`, `likeCount` |
| More, as you scroll | The `cursor` the last answer ended with | The `nextPageToken` |

The two columns differ in one way that matters: the Bluesky requests need nothing from you, and every YouTube request carries a key.

## Bluesky's Public API

Bluesky is a social network built on the AT Protocol, an open protocol: its specifications are public, and anyone may run the services that make up the network. One of those services, the **AppView**, assembles posts, profiles, and counts from across the network and answers questions about them. Bluesky runs a public AppView at `https://public.api.bsky.app/xrpc/` that answers reads for anyone, with no login ([API hosts](https://docs.bsky.app/docs/advanced-guides/api-directory)). Each method's name is part of the address: `app.bsky.actor.getProfile` reads a profile, `app.bsky.feed.getAuthorFeed` an account's posts. Like Wikipedia's API in @sec-wikipedia, it still wants to know who is asking:

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# Every request in this chapter says who is asking, as chapter 1's did; put your own address here
HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
bsky_base = "https://public.api.bsky.app/xrpc"

### Who Is Posting: A Profile

[`app.bsky.actor.getProfile`](https://docs.bsky.app/docs/api/app-bsky-actor-get-profile) takes an account's handle and returns its profile:

In [ ]:
response = requests.get(f"{bsky_base}/app.bsky.actor.getProfile",
                        params={"actor": "nws.noaa.gov"}, headers=HEADERS, timeout=30)
response.raise_for_status()
profile = response.json()
for field in ["did", "handle", "displayName", "followersCount", "followsCount", "postsCount", "createdAt"]:
    print(f"{field}: {profile[field]}")
print("verifiedStatus:", profile["verification"]["verifiedStatus"])
# did: did:plc:pfot5zyhod4cxwc5gfhzds4i
# handle: nws.noaa.gov
# displayName: National Weather Service
# followersCount: 63022
# followsCount: 4
# postsCount: 1394
# createdAt: 2025-01-17T17:50:18.044Z
# verifiedStatus: valid

The counts match the profile page, give or take the followers who arrived between the page loading and the request. Two fields deserve a closer look:

- **`did`**, a decentralized identifier, is the account's permanent ID. A handle can change; the DID doesn't, so a dataset that will be compared later should key on it.
- **`handle`**, `nws.noaa.gov`, is a domain name. An account can take a domain as its handle only by proving that it controls the domain ([Bluesky's guide](https://bsky.social/about/blog/4-28-2023-domain-handle-tutorial)), so no one but NOAA can post as `nws.noaa.gov`.

The `verification` field is a second, separate proof: Bluesky itself, as a trusted verifier, has checked the account and says so in the profile, which the website shows as a blue check ([how verification works](https://bsky.social/about/blog/04-21-2025-verification)). Accounts that look official but carry neither proof are common. A search for "National Park Service" on 2026-11-02 found "Alt National Park Service" and "The National Park Service UNCENSORED", with no `.gov` handle and no verification. Collect their posts as the agency's, and every conclusion you draw is about someone else.

### Several Accounts at Once

Five federal accounts had a `.gov` handle on Bluesky on November 2, 2026, four of them NOAA's. Others, like the Library of Congress and the Federal Reserve Bank of St. Louis, which runs FRED (@sec-government), have a `.bsky.social` handle and a verification instead. [`app.bsky.actor.getProfiles`](https://docs.bsky.app/docs/api/app-bsky-actor-get-profiles) takes up to 25 accounts in one request; `requests` repeats the `actors` parameter once for each item of a list:

In [ ]:
agencies = ["noaa.gov", "nws.noaa.gov", "climate.noaa.gov", "noaacomms.noaa.gov", "federalreserve.gov",
            "librarycongress.bsky.social", "stlouisfed.bsky.social"]
response = requests.get(f"{bsky_base}/app.bsky.actor.getProfiles",
                        params={"actors": agencies}, headers=HEADERS, timeout=30)
response.raise_for_status()
rows = []
for p in response.json()["profiles"]:
    rows.append({"handle": p["handle"], "followers": p["followersCount"], "posts": p["postsCount"],
                 "joined": p["createdAt"][:10], "verified": p["verification"]["verifiedStatus"]})
accounts = pd.DataFrame(rows).sort_values("followers", ascending=False)
print(accounts.to_string(index=False))
#                      handle  followers  posts     joined verified
#                nws.noaa.gov      63022   1394 2025-01-17    valid
#            climate.noaa.gov      46965     93 2025-01-14    valid
#                    noaa.gov      41938    150 2025-01-14    valid
# librarycongress.bsky.social       9237    551 2025-01-17    valid
#          federalreserve.gov       8405   1029 2024-12-06    valid
#          noaacomms.noaa.gov       5620    117 2025-01-14    valid
#      stlouisfed.bsky.social       1121    450 2024-12-12    valid

Most federal agencies, the Census Bureau and NASA among them, have no `.gov` handle on Bluesky, and a study of "government on Bluesky" would see the agencies that chose to come, not the government. The National Weather Service has more followers than NOAA, the agency it belongs to, and posts nearly ten times as often.

### An Account's Posts

[`app.bsky.feed.getAuthorFeed`](https://docs.bsky.app/docs/api/app-bsky-feed-get-author-feed) returns an account's posts, newest first, up to 100 a request. With `filter="posts_no_replies"` it leaves out the account's replies to others, but it still includes its **reposts**, other people's posts it shared, each marked with a `"reason"`. Those authors may be private people, so set them aside:

In [ ]:
params = {"actor": "nws.noaa.gov", "limit": 25, "filter": "posts_no_replies"}
response = requests.get(f"{bsky_base}/app.bsky.feed.getAuthorFeed", params=params, headers=HEADERS, timeout=30)
response.raise_for_status()
page = response.json()
posts = []
for item in page["feed"]:
    if "reason" in item:
        continue
    post = item["post"]
    posts.append({"created": post["record"]["createdAt"], "likes": post["likeCount"],
                  "reposts": post["repostCount"], "replies": post["replyCount"],
                  "text": post["record"]["text"][:60].replace("\n", " ")})
nws_posts = pd.DataFrame(posts)
print(len(page["feed"]), "items,", len(nws_posts), "of them the agency's own posts")
print(nws_posts.sort_values("reposts", ascending=False).head(3)[["created", "reposts", "text"]].to_string(index=False))
print("cursor:", page["cursor"])
# 25 items, 25 of them the agency's own posts
#                  created  reposts                                                         text
# 2026-10-31T18:01:13.067Z       80 Daylight Saving Time ends at 2am overnight tonight for those
# 2026-10-30T20:59:33.470Z       33 Updated Extreme Rainfall Outlook for Sunday: Another high en
# 2026-10-20T16:10:51.985Z       33 Here's a more detailed look at the expected rainfall amounts
# cursor: 2026-10-20T15:17:31.698Z

Each post carries two times. `createdAt`, inside the `record`, is when its author made it; `indexedAt`, on the post, is when the AppView saw it. They are usually seconds apart, but an author's app sets `createdAt`, so a post can be backdated: say which one you used.

### More Than One Page: the `cursor`

The answer ended with a **cursor**, and the next request sends it back to get the page that follows, as Wikipedia's `continue` did in @sec-wikipedia: you send back what the server gave you, without needing to know what it means. When there is no cursor, there is nothing more. This loop reads up to five pages of 100 posts, and keeps only what it needs from each:

In [ ]:
posts = []
cursor = None
pages = 0
while True:
    params = {"actor": "nws.noaa.gov", "limit": 100, "filter": "posts_no_replies"}
    if cursor:
        params["cursor"] = cursor
    response = requests.get(f"{bsky_base}/app.bsky.feed.getAuthorFeed", params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()
    page = response.json()
    pages += 1
    for item in page["feed"]:
        if "reason" in item:
            continue
        post = item["post"]
        posts.append({"created": post["record"]["createdAt"], "likes": post["likeCount"],
                      "reposts": post["repostCount"], "replies": post["replyCount"],
                      "embed": post["record"].get("embed", {}).get("$type", "none")})
    cursor = page.get("cursor")
    if not cursor or pages == 5:
        break
nws = pd.DataFrame(posts)
nws["created"] = pd.to_datetime(nws["created"], format="ISO8601", utc=True)
print(pages, "pages,", len(nws), "posts, from", nws["created"].min().date(), "to", nws["created"].max().date())
# 5 pages, 495 posts, from 2026-04-20 to 2026-10-31

The loop stops at five pages on purpose: the account had 1,394 posts, and the fifth page still came with a cursor. Stop on a count, a date, or the end, and say which in what you write.

### When an Agency Posts, and What Travels

The times are in UTC. Convert them to the agency's own time zone, Eastern for the Weather Service's headquarters, before asking what day or hour it posts:

In [ ]:
local = nws["created"].dt.tz_convert("America/New_York")
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_day = local.dt.day_name().value_counts().reindex(days)
ax = by_day.plot.bar(figsize=(7, 3.5), rot=0)
ax.set(title="National Weather Service posts on Bluesky by weekday, April–October 2026", ylabel="Posts", xlabel="")
plt.show()
print(by_day.to_string())
# created
# Monday        83
# Tuesday       78
# Wednesday     79
# Thursday      84
# Friday       111
# Saturday      20
# Sunday        40

The weather doesn't take weekends off, but the agency's social media staff mostly does: it posts on weekdays, most of all on Fridays, ahead of the weekend's weather. Which posts travel? Each post's `embed` says what it carries, and grouping by it compares reposts:

In [ ]:
print(nws.groupby("embed")["reposts"].agg(["count", "median"]).sort_values("count", ascending=False).to_string())
#                                 count  median
# embed
# app.bsky.embed.images             412    19.0
# app.bsky.embed.video               59    23.0
# app.bsky.embed.external            13    14.0
# app.bsky.embed.recordWithMedia      5    23.0
# app.bsky.embed.record               3    24.0
# none                                3    28.0

Almost every post carries a map or a graphic, so the comparison has little to compare: 412 of 495 are images. The medians say less than they seem to, too. A post's reach depends on the weather it warns about far more than on its format, and a comparison like this one can describe what the agency does, not say what causes a post to spread.

### Logging In, When You Need To

Everything above reads public data. Writing posts, reading your own timeline, or reading some lists needs an account: create an **app password** in Bluesky's settings, a separate password you can revoke at any time, and log in with the [`atproto` library](https://atproto.blue/) (`conda install -c conda-forge atproto`), storing the app password as you stored keys in @sec-government. For research reads, the public API needs none of it.

## YouTube Data API

YouTube's [Data API](https://developers.google.com/youtube/v3) answers anyone with a key, and counts every request against a quota. Google's documentation describes 10,000 units a day for each project, where most reads cost 1 unit, and a separate cap of 100 searches a day ([getting started](https://developers.google.com/youtube/v3/getting-started), checked 2026-11-02; the costs have changed before, so check the page when you start). This section reads channels and their uploads, which cost 1 unit a request, and never searches.

### Getting and Storing a Key

1. Sign in to the [Google Cloud console](https://console.cloud.google.com/) with a personal Google account, and create a project.
2. In the project's **API Library**, enable **YouTube Data API v3**.
3. Under **Credentials**, create an **API key**, and restrict it to the YouTube Data API v3.
4. Store it as the environment variable `YOUTUBE_KEY`, by one of the ways in @sec-government ("Setting Environment Variables"), then quit Jupyter, open a new terminal, and start Jupyter from it.

In [ ]:
import os

# Say whether the key is set, without printing it
print("YOUTUBE_KEY:", "set" if os.environ.get("YOUTUBE_KEY") else "MISSING")
youtube_key = os.environ.get("YOUTUBE_KEY")
yt_base = "https://www.googleapis.com/youtube/v3"

The key is sent as a URL parameter, so `response.url` contains it: print a response's JSON, never its URL (@sec-government, "When a Key Says `MISSING`").

### A Channel by Its Handle

[`channels.list`](https://developers.google.com/youtube/v3/docs/channels/list) finds a channel by its handle with `forHandle`, and `part` names the groups of fields you want back: `snippet` for the name and description, `statistics` for the counts, and `contentDetails` for the playlist of every video the channel has uploaded.

In [ ]:
params = {"part": "snippet,statistics,contentDetails", "forHandle": "@NOAA", "key": youtube_key}
response = requests.get(f"{yt_base}/channels", params=params, headers=HEADERS, timeout=30)
response.raise_for_status()
channel = response.json()["items"][0]
print(channel["snippet"]["title"], "|", channel["snippet"]["customUrl"])
print("subscribers:", channel["statistics"]["subscriberCount"], "| videos:", channel["statistics"]["videoCount"])
uploads = channel["contentDetails"]["relatedPlaylists"]["uploads"]

YouTube rounds `subscriberCount` down to three significant figures for every channel, so a change of a few hundred won't show. Several of this chapter's agencies have channels: `@NOAA`, `@NOAAClimate`, `@federalreserve`, `@stlouisfed`, `@uscensusbureau`, and `@NASA`. As on Bluesky, check that the agency's own website links to the channel before you treat it as theirs.

### A Channel's Videos, and Their Statistics

The uploads playlist lists every video, newest first, 50 at a time, with a `nextPageToken` to send back for the next 50. [`playlistItems.list`](https://developers.google.com/youtube/v3/docs/playlistItems/list) returns each video's ID, and [`videos.list`](https://developers.google.com/youtube/v3/docs/videos/list) takes up to 50 IDs, comma-separated, and returns their statistics:

In [ ]:
params = {"part": "contentDetails", "playlistId": uploads, "maxResults": 50, "key": youtube_key}
response = requests.get(f"{yt_base}/playlistItems", params=params, headers=HEADERS, timeout=30)
response.raise_for_status()
video_ids = []
for item in response.json()["items"]:
    video_ids.append(item["contentDetails"]["videoId"])

params = {"part": "snippet,statistics", "id": ",".join(video_ids), "key": youtube_key}
response = requests.get(f"{yt_base}/videos", params=params, headers=HEADERS, timeout=30)
response.raise_for_status()
rows = []
for v in response.json()["items"]:
    rows.append({"published": v["snippet"]["publishedAt"][:10], "title": v["snippet"]["title"][:50],
                 "views": int(v["statistics"].get("viewCount", 0)), "likes": int(v["statistics"].get("likeCount", 0))})
videos = pd.DataFrame(rows)
print(videos.sort_values("views", ascending=False).head(5).to_string(index=False))

Those two requests cost 2 units of the day's 10,000. Counts arrive as strings, as the Census Bureau's did, and a video whose owner hid its likes has no `likeCount` at all, which is why the code uses `.get()` with a default of 0. A default of 0 is a claim, though, not a fact: count how many videos had the field missing before you average it.

## Reddit After Pushshift

In 2023 Reddit priced heavy use of its API at \$0.24 per 1,000 calls (Figure 3.1), and about a month earlier it had cut off Pushshift, the volunteer-run archive that much of the research on Reddit was built on [@baumgartner2020pushshift]. (This book's author, Brian Keegan, is a co-author of that paper.) As of 2 November 2026, Pushshift answers only moderators whom Reddit has approved, and only for moderation ([Pushshift's guide](https://api.pushshift.io/guide)); anyone else gets the 403 you saw in @sec-post-api.

On 11 November 2025 Reddit ended self-service access to its Data API ([r/redditdev, 2025](https://www.reddit.com/r/redditdev/comments/1oug31u/introducing_the_responsible_builder_policy_new/)). Under its [Responsible Builder Policy](https://support.reddithelp.com/hc/en-us/articles/42728983564564-Responsible-Builder-Policy) (updated 25 March 2026), you need approval before you use any Reddit data through the API. Its researchers' section covers "academic researchers affiliated with an accredited university" doing non-commercial work, who "must re-run queries against the most recent data export to ensure findings are up-to-date with any data removals or deletions," must not keep data beyond what the project needs, and "must never attempt to re-identify, de-anonymize, or reverse engineer data about Redditors." On 28 May 2026 Reddit announced that it was "shutting down unauthenticated .json endpoints," which "can be used to scrape Reddit without accountability" ([r/modnews, 2026](https://www.reddit.com/r/modnews/comments/1tq9vxo/protecting_communities_from_scrapers_and_platform/)). The `hot.json` request in @sec-post-api now gets a block page.

Here is where Pushshift's successors stood on 2 November 2026, checked from a cloud server; a laptop on a campus network may get different answers, because some sites block cloud addresses.

| Service | Who runs it | Access | Status on 2 November 2026 |
|---|---|---|---|
| [PullPush](https://pullpush.io/) | Not named in any source this chapter could check | No key; Pushshift-style search URLs | Status 522: the server behind its Cloudflare front was not answering |
| [Arctic Shift API](https://github.com/ArthurHeitmann/arctic_shift/blob/master/api/README.md) | Arthur Heitmann, funded by donations | No key; limits change with server load | Status 200; posts are copied within seconds of being posted |
| [Arctic Shift dumps](https://github.com/ArthurHeitmann/arctic_shift/blob/master/download_links.md) | The same project, on Academic Torrents | A torrent file for each month | Monthly files through September 2026; that month alone is 74 GB |
| Watchful1's per-subreddit torrents | Watchful1, a Reddit user | Was a torrent | Gone: "They have taken down all their torrents" ([Arctic Shift issue #81](https://github.com/ArthurHeitmann/arctic_shift/issues/81), 14 October 2026) |
| [Pushshift](https://api.pushshift.io/guide) | Pushshift, with Reddit's approval | Moderators only | Status 403 |
| [ConvoKit Reddit corpora](https://convokit.cornell.edu/documentation/subreddit.html) | Cornell University | `pip install convokit`, then download | Built from Pushshift; data stops in October 2018 |

Arctic Shift's API is the one you can try. Its [documentation](https://github.com/ArthurHeitmann/arctic_shift/blob/master/api/README.md) says its rate limits depend on server load, that a status 429 means you wait the seconds in the `X-RateLimit-Reset` header, and that it offers "No uptime or performance guarantees." The request below asks for the first five posts in r/boulder in a fixed past week. Its `fields` parameter leaves out `author`, so no username reaches your computer.

In [ ]:
import requests
import pandas as pd

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}

# Arctic Shift's search for Reddit posts; it needs no key or account
url = "https://arctic-shift.photon-reddit.com/api/posts/search"
params = {
    "subreddit": "boulder",
    # A fixed past week, so the archive has finished updating these posts
    "after": "2026-10-01",
    "before": "2026-10-08",
    "sort": "asc",
    "limit": 5,
    # Ask only for the fields you need; author is left out
    "fields": "id,created_utc,retrieved_on,title,selftext,num_comments,score",
}
response = requests.get(url, params=params, headers=HEADERS, timeout=30)
response.raise_for_status()
posts = response.json()["data"]
print(response.status_code, len(posts), "posts")

df = pd.DataFrame(posts)
df["created"] = pd.to_datetime(df["created_utc"], unit="s")
# How long after posting the archive copied each post
df["archived_after"] = pd.to_timedelta(df["retrieved_on"] - df["created_utc"], unit="s")
print(df[["created", "archived_after", "score", "num_comments"]].to_string())

# Reddit puts a placeholder where a removed or deleted post's text was
gone = df["selftext"].isin(["[removed]", "[deleted]"])
print(gone.sum(), "of", len(df), "posts removed or deleted")

# Output on 2 November 2026:
# 200 5 posts
#               created  archived_after  score  num_comments
# 0 2026-10-01 00:27:03 0 days 00:00:13      8             5
# 1 2026-10-01 01:28:30 0 days 00:00:18      1             0
# 2 2026-10-01 01:59:31 0 days 00:00:25    210            12
# 3 2026-10-01 02:23:51 0 days 00:00:28      6            10
# 4 2026-10-01 03:43:46 0 days 00:00:23     14             7
# 0 of 5 posts removed or deleted

The table leaves out the titles because a post's exact words, typed into a search engine, usually lead back to its author: in health studies that quoted tweets, searching for the quote found the person 84% of the time [@ayers2018dont].

`retrieved_on` is when the archive copied a post, in seconds since 1970. Every post here was copied within half a minute, so a post deleted a minute later is already in the archive. `removed_by_category` says why a post is gone from Reddit: `deleted` if its author deleted it, and values such as `moderator` or `automod_filtered` if a moderator or AutoModerator removed it. `fields` can't select it (the API answers status 400, "'removed_by_category' is not a valid field"), but the full record, which also carries `author`, has it. Among the first 50 posts of October in r/boulder, requested that way on 2 November 2026, three had it set: one `deleted` and two `automod_filtered`. Their text reads `[deleted]` or `[removed]`, but the archive kept every title, including one that Reddit now shows as "[ Removed by moderator ]"; the archive records Reddit's version in `_meta.edited_title`.

An archive keeps what users and moderators took down; the Responsible Builder Policy asks researchers to re-run queries so that deletions are reflected. Arctic Shift removes data only when someone files its removal form. Reddit's [Data API Terms](https://redditinc.com/policies/data-api-terms) (revised 20 July 2026) forbid keeping data beyond an approved use, and its [User Agreement](https://redditinc.com/policies/user-agreement) (effective 1 July 2026) prohibits scraping "without Reddit's prior written consent." As of 2 November 2026, this chapter found no public statement from Reddit about Arctic Shift. Reviews of 727 studies that used Reddit data [@proferes2021studying] and of 134 that discussed their ethics [@fiesler2024remember] trace the field's ethical problems; the second names privacy, vulnerable populations, and unintended consequences. Deleting a post is the plainest way a person can say they no longer want it public. If you use an archive, report counts and rates, as the Additional Exercises ask, and never a deleted post's words or its author.

## Spotify: An Enclosure Case Study

Earlier versions of this chapter taught Spotify's Web API, whose audio features scored every track for `danceability`, `energy` and `valence`. On 27 November 2024 Spotify announced that new apps could no longer use Related Artists, Recommendations, Audio Features, Audio Analysis, Featured Playlists, Category Playlists, the 30-second preview URLs in multi-item responses, or algorithmic and Spotify-owned editorial playlists ([Spotify for Developers, 2024](https://developer.spotify.com/blog/2024-11-27-changes-to-the-web-api)). Apps that already had extended access kept these endpoints. Spotify's reason: "creating a more secure platform." From 15 May 2025, extended access was "reserved for apps with established, scalable, and impactful use cases" ([Spotify for Developers, 2025](https://developer.spotify.com/blog/2025-04-15-updating-the-criteria-for-web-api-extended-access)).

On 6 February 2026 Spotify said that "advances in automation and AI have fundamentally altered the usage patterns and risk profile of developer access," and that Development Mode, the tier every new app starts in, was a sandbox that "should not be relied on as a foundation for building or scaling a business" ([Spotify for Developers, 2026](https://developer.spotify.com/blog/2026-02-06-update-on-developer-access-and-platform-security)). Its [migration guide](https://developer.spotify.com/documentation/web-api/tutorials/february-2026-migration-guide) (checked 2 November 2026) applied new rules to new Development Mode apps on 11 February 2026, and to existing ones on 9 March. The app's owner must have Premium: "If the owner's Premium subscription lapses, the app will stop working." A new app may have up to five users. Spotify removed an artist's top tracks, the batch endpoints that fetched several tracks, albums or artists in one call, and the `followers` and `popularity` fields. Search now returns at most 10 results per request, down from 50, and 5 by default.

None of this is a per-call price like Reddit's. These are product decisions announced on a developer blog: the 2024 removals on the day they took effect, and the 2026 rules five days ahead. A study built on audio features in 2023 can't be rebuilt by a student who starts today. As of November 2026, a student who registers an app can look up one track at a time, see ten search results at a time, and do either only while paying for Premium.

## Comparing Platforms

| | Bluesky | YouTube | Reddit, through an archive | Spotify |
|---|---|---|---|---|
| Who controls access | No single company; Bluesky runs the public AppView | Google | The archive's maintainer, against Reddit's terms | Spotify |
| What you need | Nothing | A key from a Google account | Nothing, for now | Premium, and fewer endpoints |
| Limits | Rate limits, generous for reads | 10,000 units and 100 searches a day | Changes with the archive's load | Search capped at 10 results |
| Deleted content | Gone from the API | Gone from the API | Often kept | — |
| Stable enough to build on? | The protocol is open | As long as the key and quota hold | No guarantee | No |

The last two rows pull in different directions. An open protocol makes a platform's data durable for research, and an archive that keeps deleted posts makes a study reproducible. But a person who deleted a post expected it to be gone, and an archive that keeps it keeps it against their wishes.

## Recommended Exercises

This guided exercise is the chapter's take-home assignment. Work through it in the companion notebook, filling in each empty code cell, and submit the completed notebook. The steps build on one another, so do them in order; everything you need appears in this chapter or an earlier one.

You will compare how two federal accounts on Bluesky used the platform in September 2026. Pick two from this chapter's list, or another verified institutional account, never a private person's.

**Step 1 — Set up, and check who you are asking about.** Define your `HEADERS` with your own email, and request both accounts' profiles with `getProfiles`. Print each handle, its `verifiedStatus`, and whether the handle is a domain, and say in a comment why you trust that each account is the institution's.

In [ ]:
# Step 1: Define HEADERS, request both profiles with getProfiles, and print each handle, its verifiedStatus, and whether the handle ends in .gov
# Your code here

**Step 2 — Collect one account's September posts.** Page through the first account's feed with `getAuthorFeed` and its `cursor`, skipping reposts, until you reach posts older than September 1, 2026. Keep the posts created in September, with their time, likes, reposts, replies, and embed type.

In [ ]:
# Step 2: Page through the first account's feed until you pass September 1, 2026, and keep its own September posts in a DataFrame
# Your code here

**Step 3 — Describe the month.** Convert the times to the account's time zone, and compute: the number of posts, posts by weekday, the share of posts with an image or a video, and the median reposts.

In [ ]:
# Step 3: Compute the number of posts, posts by weekday, the share with an image or video, and the median reposts
# Your code here

**Step 4 — Repeat for the second account.** Collect and describe the second account's September the same way.

In [ ]:
# Step 4: Collect and describe the second account's September posts
# Your code here

**Step 5 — Compare.** Put the two accounts' numbers side by side in one DataFrame, and plot one comparison you find informative, such as posts by weekday for both.

In [ ]:
# Step 5: Build the comparison DataFrame and plot one comparison
# Your code here

**Step 6 — Interpret.** In four to six sentences: how did the two accounts use the platform differently in September? What would you need to know to explain the difference, rather than describe it? What can't the public API tell you, such as who saw a post, and which other source would you need for it?

In [ ]:
# Step 6: Write your answer here as comments, or convert this cell to Markdown.

## Additional Exercises

These are open-ended extensions, with no scaffold and no fixed path.

1. **The same agencies on YouTube.** For three of the chapter's agencies, read each channel's last 50 uploads and their statistics. How do views compare with the agency's Bluesky followers? Report what your collection cost in quota units, and how you'd collect a year of uploads within one day's quota.

2. **What an archive keeps.** Using the Arctic Shift API, collect a month of posts from a subreddit about your town, leaving out authors. What share was later removed, and by whom (`removed_by_category`)? Report counts only, and say how your findings would change if the archive honored deletions.

3. **Open and keyed.** Pick one agency with an account on both Bluesky and YouTube. Compare what each API let you learn about it, what each asked of you, and what each would let you share in a public repository under its terms.

4. **Graduate extension (INFO 5617).** Read @bruns2019after and @davidson2023platform. Then design (do not execute) a measurement plan for studying how federal agencies communicate during an emergency, such as a hurricane, on one open and one keyed platform from this chapter. For each, specify the endpoints and fields, the sampling frame, what the platform's access rules prevent you from observing, and how those constraints could bias your conclusions. Then say whether research access under the European Union's Digital Services Act (@sec-post-api) would change any of it for a study like this one, and why.

## Social History and Public Interest

The high-water mark of platform data for research came in January 2021, when Twitter opened its Academic Research track: free full-archive search, up to 10 million tweets a month, for approved non-commercial projects ([TechCrunch, 2021](https://techcrunch.com/2021/01/26/twitters-new-api-platform-now-opened-to-academic-researchers/)). Two years later, after the company's sale, researchers learned that free access was ending ([Ledford, 2023](https://www.nature.com/articles/d41586-023-00460-z)), and studies built on it became difficult or impossible to repeat [@davidson2023platform]. Freelon had named the condition in 2018, the "post-API age" [@freelon2018computational], after Facebook and Twitter, in the aftermath of the Cambridge Analytica controversy, severely restricted researchers' access to their APIs [@bruns2019after].

Reddit's research data ran on volunteers. Pushshift began in 2015 with a dump advertised as "every publicly available Reddit comment," and its archive became a common source for Reddit research [@baumgartner2020pushshift] (disclosure: the book's author is a co-author of that paper). Archives built by volunteers are not complete: an early check of the 2015 comment dump found large-scale gaps that could bias studies built on it [@gaffney2018caveat]. They are also fragile: of the successors in this chapter's table, the one that worked depends on one person's server and on Reddit's tolerance.

Regulation offers a different route. Article 40 of the European Union's Digital Services Act requires very large online platforms and search engines to give vetted researchers access to data for research on systemic risks in the EU (Figure 3.2). The Commission adopted the delegated act that sets the procedure on 2 July 2025 ([European Commission, 2025](https://digital-strategy.ec.europa.eu/en/news/commission-adopts-delegated-act-data-access-under-digital-services-act)). It was published as [Delegated Regulation (EU) 2025/2050](https://eur-lex.europa.eu/eli/reg_del/2025/2050/oj/eng) on 9 October 2025 and entered into force twenty days later, on 29 October. A researcher applies to a Digital Services Coordinator, the national regulator, and must show an affiliation with a research organisation, independence from commercial interests, and the ability to keep data secure, disclose the project's funding, and commit to publishing the results. A new data access portal handles the messages among researchers, platforms and regulators. Access rests on law rather than on a platform's goodwill or a volunteer's server, and only researchers whom a regulator vets get it.

This chapter's running case is federal agencies' public accounts. When Fiesler and Proferes surveyed Twitter users, few knew that researchers used public tweets, and most felt researchers should not use them without permission [@fiesler2018participant]. A federal agency's account is different: it speaks for a public body, in public, as part of its job, and it has no privacy to lose. Quoting its posts identifies no private person, and when an agency deletes a post, it changes a public record.

## Common Issues to Debug

- **An account that isn't who it seems.** Check the handle and the `verification` field before you collect; a familiar name is not a proof.
- **A repost in your data.** `getAuthorFeed` includes reposts, with a `"reason"`. Skip them, or you will have collected other people's posts, perhaps private people's, under the agency's name.
- **A loop that never ends, or ends early.** Stop when the answer has no `cursor`, or on a count or a date you chose, and say which.
- **Times in the wrong zone.** Bluesky's and YouTube's times are UTC. Convert them before counting by day or hour.
- **`KeyError` or `MISSING` for `YOUTUBE_KEY`.** Jupyter never received the variable: quit Jupyter, open a new terminal, and start it again (@sec-government, "When a Key Says `MISSING`").
- **YouTube's `403` with `quotaExceeded`.** The project used its 10,000 units, or its 100 searches, for the day; the quota resets at midnight Pacific time. Read channels and playlists instead of searching.
- **YouTube's counts as strings, or missing.** Convert `viewCount` and `likeCount` with `int()`, and count the videos where a field is absent before treating it as zero.
- **An archive that doesn't answer.** Archives of Reddit data run on volunteers' servers, with no guarantee: a `522` or a timeout is the service, not your code. Save what you collect, and cite the date.

## Key Takeaways

A platform's API exists at the platform's discretion, and since 2023 most platforms have narrowed theirs, so date what you collect and say what access you had. Check who an account belongs to before you collect it. Bluesky's open protocol answers anyone, YouTube's keyed API counts what you use, and Reddit's data now lives mostly in archives that keep what users deleted. Whichever you use, social data is people data: read public institutions, keep private people out of what you save and show, and prefer the source whose terms you can follow.

## Further Reading

- Bluesky API documentation: <https://docs.bsky.app/>
- AT Protocol: <https://atproto.com/>
- YouTube Data API: <https://developers.google.com/youtube/v3>
- Arctic Shift: <https://github.com/ArthurHeitmann/arctic_shift>
- Spotify Web API, February 2026 migration guide: <https://developer.spotify.com/documentation/web-api/tutorials/february-2026-migration-guide>
- @baumgartner2020pushshift — the Pushshift Reddit dataset
- @proferes2021studying — how researchers have studied Reddit, and the ethics of it
- @bruns2019after — platforms versus scholarly research after the "APIcalypse"